In [2]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# IFS long-run spending as % of GDP. Health, Pensioner social security, PS net debt interest.
# 1978-79 to 2022-23 (overlap window where all three are populated).
data = [
 (1978,3.9,5.0,3.5),(1979,3.8,4.8,3.6),(1980,4.1,5.0,3.8),(1981,4.2,5.3,3.9),
 (1982,4.1,5.4,3.6),(1983,4.0,5.4,3.5),(1984,4.0,5.3,3.6),(1985,3.9,5.3,3.4),
 (1986,3.9,5.3,3.3),(1987,4.0,5.0,3.0),(1988,4.0,4.7,2.6),(1989,3.9,4.6,2.3),
 (1990,4.0,4.8,2.1),(1991,4.3,5.1,1.8),(1992,4.6,5.4,2.0),(1993,4.7,5.5,2.2),
 (1994,4.8,5.4,2.4),(1995,4.8,5.3,2.6),(1996,4.6,5.2,2.6),(1997,4.6,5.2,2.7),
 (1998,4.6,5.2,2.5),(1999,4.7,5.3,2.0),(2000,4.9,5.3,2.0),(2001,5.2,5.5,1.8),
 (2002,5.5,5.4,1.7),(2003,5.9,5.4,1.7),(2004,6.2,5.5,1.8),(2005,6.3,5.5,1.7),
 (2006,6.4,5.4,1.9),(2007,6.5,6.0,1.8),(2008,6.9,5.9,2.0),(2009,7.5,6.4,1.9),
 (2010,7.4,6.4,2.6),(2011,7.3,6.4,2.6),(2012,7.2,6.6,2.2),(2013,7.2,6.3,2.1),
 (2014,7.1,6.3,1.8),(2015,7.1,6.2,1.8),(2016,7.0,6.0,1.8),(2017,7.0,5.9,2.0),
 (2018,7.0,5.8,1.6),(2019,7.3,5.7,1.4),(2020,10.5,6.3,0.9),(2021,9.2,5.7,2.2),
 (2022,8.4,5.6,3.8),
]
df = pd.DataFrame(data, columns=["year","Health","Pensions","Debt interest"])
long = df.melt("year", var_name="cat", value_name="pct")
long["date"] = pd.to_datetime(long["year"], format="%Y")

order = ["Health","Pensions","Debt interest"]
# Health + pensions = rising public goods (teal/blue); debt interest = the squeeze (red)
palette = {"Health":"#36B7B4","Pensions":"#0063AF","Debt interest":"#E6224B"}
color = alt.Color("cat:N", scale=alt.Scale(domain=order,
    range=[palette[c] for c in order]), legend=None)

lines = alt.Chart(long).mark_line(strokeWidth=2).encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=8), title=None),
    y=alt.Y("pct:Q", scale=alt.Scale(domain=[0,11]), title="Spending (% of GDP)",
            axis=alt.Axis(labelExpr="datum.label + '%'")),
    color=color, detail="cat:N")
labels = alt.Chart(long).mark_text(align="left", dx=6, fontSize=12, fontWeight="bold").encode(
    x=alt.X("date:T", aggregate="max"),
    y=alt.Y("pct:Q", aggregate={"argmax":"date"}),
    text="cat:N", color=color)

caption = alt.Title(
    text="Source: IFS long-run public finances; OBR",
    subtitle=[
        "UK public spending by function, % of GDP, 1978–2022.",
        "The demands that grow — health and pensions — climb steadily, while debt interest rises again.",
    ],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (lines + labels)
    .properties(width=700, height=360, padding={"right":90}, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".",name="spending_composition", svg=True)
chart

alt.LayerChart(...)